# Land-use-adjusted ecological consistency

This notebook evaluates clustering results using the land-use-adjusted ecological consistency metric now moved into `moduli.vegetation_eval`.

The metric separates:

1. **Agricultural consistency** — measured inside agricultural land-cover pixels using detailed CLC agriculture classes.
2. **Natural/ecological consistency** — measured inside natural/ecological pixels, including forests, shrubland, grassland/bare areas, wetlands and water.
3. **Artificial share** — reported separately and excluded from the score.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from moduli.vegetation_eval import (
    load_clc_legend,
    attach_node_labels_to_pixels,
    merge_clusters_with_vegetation,
    add_clc_metadata,
    vegetation_contingency_table,
    land_use_component,
    agriculture_class_label,
    natural_class_label,
    entropy_quality_for_subset,
    land_use_adjusted_ecological_consistency,
    overall_land_use_summary,
    evaluate_label_set,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## Paths

In [ ]:
VEGETATION_PATH = Path("../vegetation.csv")
LEGEND_PATH = Path("../Vegetacija/u2018_clc2018_v2020_20u1_raster100m/Legend/CLC2018_CLC2018_V2018_20_QGIS.txt")


## Load vegetation data and legend

In [ ]:
vegetation = pd.read_csv(VEGETATION_PATH)
legend = load_clc_legend(LEGEND_PATH)

print("Vegetation shape:", vegetation.shape)
display(vegetation.head())

print("Legend shape:", legend.shape)
display(legend.head())


## Attach cluster labels

Use one of the two options below.

### Option A: labels are already attached to pixels

Use this if you already have a dataframe with columns `row`, `col`, and `cluster`.

### Option B: labels are assigned to model nodes

Use this if your clustering was performed on superpixels. Then you need:

- `data["df"]` from `prepare_model_data`;
- `data["node_col"]`;
- `labels` from the clustering algorithm.


In [ ]:
# ---------------------------------------------------------------------
# OPTION A
# ---------------------------------------------------------------------
# df_clusters = pd.read_csv("clusters_pixel_level.csv")
# Expected columns: row, col, cluster


# ---------------------------------------------------------------------
# OPTION B
# ---------------------------------------------------------------------
# df_clusters = attach_node_labels_to_pixels(
#     df=data["df"],
#     node_labels=labels,
#     node_col=data["node_col"],
#     cluster_col="cluster",
# )


## Merge clusters with vegetation

In [ ]:
# merged = merge_clusters_with_vegetation(
#     clusters_df=df_clusters,
#     vegetation_df=vegetation,
#     cluster_col="cluster",
#     row_col="row",
#     col_col="col",
#     class_col="clc_class",
#     nodata_values=(0, 999),
#     how="inner",
# )

# merged = add_clc_metadata(merged, legend=legend, class_col="clc_class")
# merged["land_use_component"] = merged.apply(land_use_component, axis=1)
# merged["agriculture_class"] = merged.apply(agriculture_class_label, axis=1)
# merged["natural_class"] = merged.apply(natural_class_label, axis=1)

# display(merged.head())
# print(merged["land_use_component"].value_counts(dropna=False))


## Run the metric

Higher `combined_ecological_consistency` is better.

Artificial land is excluded from the score but reported separately.


In [ ]:
# per_cluster_quality = land_use_adjusted_ecological_consistency(
#     merged,
#     cluster_col="cluster",
# )

# display(per_cluster_quality.sort_values("combined_ecological_consistency", ascending=False))

# overall = overall_land_use_summary(per_cluster_quality)
# overall


## Diagnostics

In [ ]:
# Cluster x broad land-cover group distribution
# display(
#     vegetation_contingency_table(
#         merged,
#         cluster_col="cluster",
#         class_col="clc_group",
#         normalize="index",
#     )
# )

# Cluster x land-use component distribution
# display(
#     vegetation_contingency_table(
#         merged,
#         cluster_col="cluster",
#         class_col="land_use_component",
#         normalize="index",
#     )
# )

# Cluster x detailed agriculture class distribution
# agri_only = merged[merged["land_use_component"] == "agriculture"]
# display(
#     vegetation_contingency_table(
#         agri_only,
#         cluster_col="cluster",
#         class_col="agriculture_class",
#         normalize="index",
#     )
# )


## Comparing multiple clustering methods

If you have several label arrays, store them in a dictionary and evaluate them in a loop.

Example:

```python
label_sets = {
    "KMeans": kmeans.labels_,
    "ACO": aco.labels_,
    "ACO + refinement": aco_refinement.labels_,
}
```


In [ ]:
# summaries = []
# per_cluster_by_method = {}

# for name, labels in label_sets.items():
#     overall, per_cluster, merged_method = evaluate_label_set(
#         method_name=name,
#         labels=labels,
#         data_df=data["df"],
#         node_col=data["node_col"],
#         vegetation=vegetation,
#         legend=legend,
#     )
#     summaries.append(overall)
#     per_cluster_by_method[name] = per_cluster

# summary_df = pd.DataFrame(summaries)
# display(summary_df.sort_values("weighted_combined_ecological_consistency", ascending=False))
